# Intermediate
## Calcul du Fatigue Index

**Objectif :** Calculer le Fatigue Index (FI), indicateur composite de fatigue 
physiologique avant chaque match.

**Pourquoi une couche intermédiaire ?**
Ce calcul est trop complexe pour le staging et trop générique pour un mart spécifique.
Il est réutilisé par `mart_physical_condition` et `mart_game`.

**Formule SportMetrics :**

| Composante | Poids | Détail |
|---|---|---|
| Charge interne | 30% | 60% FC normalisée + 40% niveau fatigue |
| Charge externe | 40% | 70% intensité charge + 30% volume hebdo |
| Récupération | 30% | Inverse du score de récupération |

**Clé :** `player_id + session_id`  
**Granularité :** 1 ligne = 1 session d'entraînement par joueur

In [0]:
-- ============================================
-- Intermediate : int_fatigue_index
-- Calcul en 3 étapes :
-- 1. Normalisation des métriques (0→1)
-- 2. Calcul du Fatigue Index pondéré
-- 3. Interprétation + projection avant match
-- ============================================
CREATE OR REPLACE TABLE sport_metrics.mart.int_fatigue_index AS

WITH sts AS (
    SELECT * FROM sport_metrics.staging.team_training_sessions
),
tps AS (
    SELECT * FROM sport_metrics.staging.team_players_stats
),
spi AS (
    SELECT * FROM sport_metrics.staging.team_players_personal_info
),
normalisation AS (
    SELECT
        sts.season,
        sts.session_date,
        sts.session_id,
        sts.player_id,
        sts.next_match_id,
        tps.minutes_played,
        sts.recovery_time_hours,
        sts.days_before_match,
        -- Normalisation FC : ratio FC / FC max théorique (220 - âge)
        (sts.heart_rate / (220 - spi.age))                                      AS hr_norm,
        -- Normalisation fatigue : Low=1/3 · Medium=2/3 · High=3/3
        (CASE
            WHEN sts.fatigue_level = 'Low'    THEN 1
            WHEN sts.fatigue_level = 'Medium' THEN 2
            ELSE 3
        END) / 3.0                                                               AS fatigue_level_norm,
        -- Score récupération : ratio temps disponible / temps repos nécessaire
        ROUND((sts.days_before_match * 24) / sts.recovery_time_hours, 2)        AS recovery_score,
        sts.load_intensity_score / 10.0                                          AS load_intensity_norm,
        -- Volume hebdo normalisé par rapport au max équipe
        sts.weekly_training_hours / ((MAX(sts.duration_min) OVER () / 60) * 7)  AS weekly_training_norm
    FROM sts
    JOIN spi ON spi.player_id = sts.player_id
    JOIN tps ON sts.player_id = tps.player_id
            AND sts.next_match_id = tps.game_id
    WHERE tps.minutes_played >= 5  -- exclusion garbage time
),
fatigue_calc AS (
    SELECT
        season,
        session_date,
        session_id,
        player_id,
        next_match_id,
        recovery_score,
        -- FORMULE SPORTMETRICS
        (0.30 * (0.6 * hr_norm + 0.4 * fatigue_level_norm)
       + 0.40 * (0.7 * load_intensity_norm + 0.3 * weekly_training_norm)
       + 0.30 * (1 - LEAST(1, recovery_score))) * 100                           AS fatigue_index_score
    FROM normalisation
),
fatigue_index_fi AS (
    SELECT
        f.season,
        f.session_date,
        f.session_id,
        f.next_match_id,
        f.player_id,
        f.recovery_score,
        ROUND(f.fatigue_index_score, 2)                                          AS fatigue_index_score,
        CASE
            WHEN f.fatigue_index_score <= 30 THEN 'Fraîcheur optimale'
            WHEN f.fatigue_index_score <= 50 THEN 'Fatigue légère'
            WHEN f.fatigue_index_score <= 65 THEN 'Fatigue modérée'
            WHEN f.fatigue_index_score <= 80 THEN 'Fatigue élevée / Risque'
            ELSE 'Danger blessure / baisse performance'
        END                                                                      AS fi_interpretation,
        CAST(CASE
            WHEN n.recovery_time_hours > n.days_before_match * 24
            THEN (n.recovery_time_hours - n.days_before_match * 24)
            ELSE 0
        END AS INT)                                                              AS recovery_needed_hours
    FROM fatigue_calc f
    JOIN normalisation n USING (player_id, session_id)
)
SELECT
    f.season,
    f.session_date,
    f.session_id,
    f.player_id,
    n.next_match_id,
    f.recovery_score,
    f.fatigue_index_score,
    f.fi_interpretation,
    f.recovery_needed_hours,
    -- Fi_before_match : fatigue résiduelle estimée au coup d'envoi
    ROUND(LEAST(100, (
        f.fatigue_index_score * f.recovery_needed_hours / (n.days_before_match * 24)
    )), 2)                                                                       AS fi_before_match,
    current_timestamp()                                                          AS updated_at
FROM fatigue_index_fi f
JOIN normalisation n USING (player_id, session_id);

In [0]:
-- ============================================
-- Vérification
-- ============================================
SELECT
    COUNT(*)                                    AS nb_lignes,
    COUNT(DISTINCT player_id)                   AS nb_joueurs,
    ROUND(AVG(fatigue_index_score), 2)          AS fi_moyen,
    ROUND(MIN(fatigue_index_score), 2)          AS fi_min,
    ROUND(MAX(fatigue_index_score), 2)          AS fi_max,
    COUNT(CASE WHEN fi_interpretation = 'Danger blessure / baisse performance' 
          THEN 1 END)                           AS nb_danger
FROM sport_metrics.mart.int_fatigue_index;